
# SVS Ensemble Simulation for E1 Enclosure with Perturbed Meteorology and Parameters

This script sets up and executes an ensemble simulation of the SVS model for the E1 enclosure.

## Methodology

1. **Initialization:**
   - Imports necessary libraries.
   - Defines file paths for input data, model executable, working directory, and output files.
   - Defines data structures (`SoilType`, `Enclosure`, `SiteParams`, `SVSParams`) to organize SVS model parameters.
   - Verifies that all required file paths exist.

2. **Input Object Creation:**
   - Specifies soil properties for topsoil and cover material layers.
   - Defines the layering structure of the soil column for the E1 enclosure.
   - Sets site-specific parameters (e.g., latitude, longitude, slope) and SVS model parameters.
   - Initializes a `ModelInputData` object that encapsulates the configuration for the SVS model.

3. **SVS Test Run (Optional):**
   - Creates an instance of the `SVSModel` class using the default input configuration.
   - Uncomment and run `svs_test.run_svs()` to perform a test simulation with the default setup.

4. **Ensemble Run:**
   - Loads a list of perturbed meteorological (.met) files from the `met_ensemble_dir`.
   - Loads a dictionary of perturbed parameter sets from a pickle file.
   - Initializes a `PerturbAndRun` object to manage the ensemble simulation.
   - Uncomment and run the following lines to initiate the ensemble:
     - `ensemble_run.create_param_scen_df()` (Creates a dataframe of parameter scenarios)
     - `ensemble_run.dfscenarios.to_csv(...)` (Saves the parameter scenarios)
     - `ensemble_run.run_all_parallel(...)` (Executes the ensemble simulation in parallel)

## File Paths

* **Input Data:**
    - Hourly meteorological data: `../Data/OriginalForcingData.csv`
    - Default .met file: `../Data/basin_forcing.met`
    - SVS executable: `../SVS_Source_Code/svs_Jun12_2024`
    - Meteorological ensemble directory: `./Results/Met_Ens/`
    - Parameter ensemble dictionary: `./Results/30scenarios_5params_Mar13.pickle`

* **Output Data:**
    - Parameter scenarios: `./Results/dfscenarios_ens_run_Mar11.csv`
    - SVS model output (for each scenario): saved as `.feather` files in the `working_dir`

## Parameters

- `N_PROCS`: Maximum number of processors for parallelization (default: 4).
- `HOST_DIR_NAME`: Name of the directory containing the enclosure simulation files (default: "test_run").
- `START_DATE`, `END_DATE`, `SPINUP_END_DATE`, `SPINUP_END_DATE_TZ`: Simulation time settings.

## Dependencies

- **Python libraries:** `pathlib`, `collections`, `pickle`, `svspyed` (custom package)

## Key Assumptions

- The perturbed meteorological and parameter datasets are compatible with the SVS model.
- The `svspyed` package is correctly installed and provides the necessary functionality.

## Author

Alireza Amani

## Date

2024-07-23

In [1]:
# 0) Notes ---------------------------------------------------------------------
'''
In this notebook, we prepare and run the ensemble simulation considering
M number of meteo scenarios and N number of parameters scenarios.

'''
# ______________________________________________________________________________

# 1) Required Libraries --------------------------------------------------------
from pathlib import Path
from collections import namedtuple
import pickle

from svspyed.utils.helper_functions import (
    check_csv_columns, get_layering_df, populate_layering_df,
)
from svspyed.input_preparation.prep_svs import ModelInputData
from svspyed.model.svs_model import SVSModel
from svspyed.ensemble_run.perturb_run import PerturbAndRun
# ______________________________________________________________________________

# 2) Variables -----------------------------------------------------------------

## (max) number of processors to use for parallelization
N_PROCS = 4

# relevant paths
paths = dict(
    # input meteorological data required to drive SVS
    hourly_meteo = Path(
        "../Data/OriginalForcingData.csv"
    ),

    # default .met file
    default_met_file = Path('../Data/basin_forcing.met'),

    # SVS executable (compiled from Fortran source code)
    svs_exec = Path(
        "../SVS_Source_Code/svs_Jun12_2024"
    ),

    # the working dir where the run-time files will be stored
    working_dir = Path(
        "./runs/"
    ),

    # .met ensemble dir
    met_ensemble_dir = Path(
        "./Results/Met_Ens/"
    ),

    # param ens dict
    param_ensemble_dict = Path('./Results/30scenarios_5params_Mar13.pickle'),
)

## 2.2. SVS related variables

### 2.2.1. a container for the properties of a soil type
SoilType = namedtuple("SoilType", [
    'code',
    'sand', 'clay',
    'wsat', 'wfc', 'wwilt', 'wunfrz',
    'ksat',
    'psisat', 'bcoef',
    # 'rhosoil',
])

### 2.2.2. a container for a soil cover
Enclosure = namedtuple("Enclosure", [
    'layering',
    'soil_layers'
])

### 2.2.3. a container for site-specific parameters required to configure SVS
SiteParams = namedtuple("SiteParams", [
    'deglat', 'deglng',
    'slop', 'zusl', 'ztsl',
    'observed_forcing', 'draindens', 'vf_type'
])

### 2.2.4. a container for SVS internal parameters
SVSParams = namedtuple("SVSParams", [
    'SCHMSOL', 'lsoil_freezing_svs1', 'soiltext', 'read_user_parameters',
    'save_minutes_csv', 'water_ponding', 'KFICE', 'OPT_SNOW', 'OPT_FRAC',
    'OPT_LIQWAT', 'WAT_REDIS', 'tperm', 'user_wfcdp',
    "OPT_VEGFLUX", "OPT_AGGFLUX", "OPT_VEGCOND",
    "user_VEGDAT_INDEX", "user_VEGDAT_VALUE",
    "user_LAM_VEGL_STAB", "user_LAM_VEGL_UNSTAB",
])

# checks #
# check if the paths lead to existing files/directories
for key, path in paths.items():
    if not path.exists():
        raise FileNotFoundError(f"{key} path does not exist: {path}")

# ______________________________________________________________________________

# 3) Input Object --------------------------------------------------------------
## 3.1. soil properties
TPsoil = SoilType(
    code = "Topsoil",
    sand = 74.8, # percent
    clay = 0, # percent
    wsat = 0.37, wfc = 0.10, wwilt = 0.032, wunfrz = 0.08, # volumetric fraction
    ksat = 1.416e-05, # m.s-1
    psisat = 0.24, bcoef = 3.6, # mH2O, unitless
)
CMsoil = SoilType(
    code = "CoverMaterial",
    sand = 68.0, # percent
    clay = 7.5, # percent
    wsat = 0.367, wfc = 0.10, wwilt = 0.01, wunfrz=0.04, # volumetric fraction
    ksat = 3.58e-06, # m.s-1
    psisat = 0.4, bcoef = 3.43, # mH2O, unitless
)


## 3.2. soil column
E1 = Enclosure(
    layering    = "15:2.5:Topsoil + 160:5:CoverMaterial + 15:2.5:CoverMaterial",
    soil_layers = [TPsoil, CMsoil],

    # "190:5:CM" -> 190 cm total depth, divided into 5 cm increments, each with
    # the properties of CMsoil
    # could be changed to, for example, `15:2.5:CM + 175:5:CM` to have a 15 cm
    # top layer with 2.5 cm increments, and a 175 cm bottom layer with 5 cm
)

## 3.3. parameters describing the site, identical for all enclosures
site_params = SiteParams(
    deglat = 45.82, deglng = -72.37, # degrees latitude and longitude

    slop = 0.02, zusl = 10.0, ztsl = 1.5, # slope, heights for momentum and temp
    observed_forcing = "height", draindens = 0.0, vf_type = 13
    # drainage density, vegetation fraction type
)

## 3.4. parameters specific to the SVS model
### check SVS source code and doc for more info on these parameters
svs_params = SVSParams(
    SCHMSOL = "SVS", lsoil_freezing_svs1 = ".TRUE.", soiltext = "NIL",
    read_user_parameters = 1, save_minutes_csv = 0, water_ponding = 0,
    KFICE = 0, OPT_SNOW = 2, OPT_FRAC = 1, OPT_LIQWAT = 1, WAT_REDIS = 0,
    tperm = 280.15, user_wfcdp = 0.36,
    OPT_VEGFLUX = 3, OPT_AGGFLUX = 2, OPT_VEGCOND = 2,
    user_VEGDAT_INDEX = 13, user_VEGDAT_VALUE = 0.95,
    user_LAM_VEGL_STAB = 30.0, user_LAM_VEGL_UNSTAB = 4.4,
    # user_D50_INDEX = 13, user_D50_VALUE = 0.5,
    # user_D95_INDEX = 13, user_D95_VALUE = 1.5,
)

## 3.5. name of the directory containing the enclosure
HOST_DIR_NAME = "test_run"

## 3.6. mapping of the required meteo variables to the labels in the forcing file
## needed for the `save_csv_as_met` function
meteo_cols = {
    "utc_dtime": "datetime_utc",
    "air_temperature": "Air Temperature (degC)",
    "precipitation": "Precipitation (mm)",
    "wind_speed": "Wind Speed (m/s)",
    "atmospheric_pressure": "Atmospheric Pressure (Pa)",
    "shortwave_radiation": "Shortwave Radiation (W/m2)",
    "longwave_radiation": "Longwave Radiation (W/m2)",
    "specific_humidity": "Specific Humidity (kg/kg)",
    "relative_humidity": "Relative Humidity (%)"
}

### 3.6.1. check if the columns in the meteo file are correct
print("Checking meteo file columns...")
check_csv_columns(paths["hourly_meteo"], meteo_cols);

## 3.7. simulation start_date
START_DATE = "2017-182-04-00"
END_DATE = ""

## 3.8. spinup end date
SPINUP_END_DATE = "2018-07-01 00:00:00"

## 3.9. spinup date timezone
SPINUP_END_DATE_TZ = "America/Montreal"

## 3.10. name of the parameters to assign for each layer: all fields except `code`
layer_params = tuple(field for field in CMsoil._fields if field != 'code')

## 3.11. read layering information and populate the dataframe
dfenc = get_layering_df(E1)
dfenc = populate_layering_df(dfenc, E1, site_params)

## 3.12. initialize the input object
input_object = ModelInputData(
    work_dir_path=paths["working_dir"],
    soilcover_info=dfenc,
    metfile_path=paths["hourly_meteo"],
    exec_file_path=paths["svs_exec"],
    host_dir_name=HOST_DIR_NAME,
    meteo_col_names=meteo_cols,
    param_col_names={
        **{k: k for k in layer_params},
        **{k: k for k in site_params._fields},
    },
    model_params=svs_params._asdict(),
    start_date=START_DATE,
    end_date=END_DATE,
    spinup_end_date=SPINUP_END_DATE,
    time_zone=SPINUP_END_DATE_TZ,
    model_tsetp=1,
    copy_metfile=paths["default_met_file"],
)
# ______________________________________________________________________________

# 4) SVS test run --------------------------------------------------------------

## 4.1. initialize the model
svs_test = SVSModel(input_object, True, True)

## 4.2. run the model
# svs_test.run_svs()

# ______________________________________________________________________________

# 5) Perturb and Run -----------------------------------------------------------

## met files
met_files = list(paths["met_ensemble_dir"].glob("*.met"))

## param files
with open(paths["param_ensemble_dict"], "rb") as f:
    param_ensemble = pickle.load(f)

## initialize the perturb and run object
ensemble_run = PerturbAndRun(
    svs_default_input=input_object,
    parameter_scenarios=param_ensemble,
    met_paths=met_files,
    njobs=N_PROCS,
    verbose=True
)

##  ----- uncomment the below lines to run the ensemble simulation
## after the run is complete, `ensemble_run` will have the resulting (large) dataframe
## as an attribute. Save it to a csv file for later use.
## also, model outputs are stored for each scenario as a .feather file in the
## working directory (i.e. checkpointing) -----

# ensemble_run.create_param_scen_df()
# ensemble_run.dfscenarios.to_csv('./Results/dfscenarios_ens_run_Mar11.csv', index=False)
# ensemble_run.run_all_parallel(output_time_scale="hourly", effort_id="ensemble_run_Mar11_")

# ______________________________________________________________________________


Checking meteo file columns...
All columns found!

a folder named `test_run` is created inside the dir: `runs`


The `MESH_input_soil_levels.txt` file is created inside the host folder located at 
dir: `runs/test_run`.
In this file 44 soil layers are defined.

.met file copied to: runs/test_run/basin_forcing.met

The MESH_input_run_options.ini is created inside the host folder
at: `runs/test_run`
Simulatuin start time (UTC): 2017-07-01 04:00:00
Simulatuin end time (UTC): 2022-12-31 04:00:00

MESH_parameters.txt created in the host folder at: `runs/test_run`

The SVS executable file is copied into the host folder. named `SVS_exe`.

SVSModel instance created.


Number of scenarios: 900

